In [1]:
import torch
import torch.nn as nn

In [ ]:
# input & output initialization
x = torch.tensor([
    [1, 2, 3, 4, 5],
    [2, 3, 4, 5, 6],
    [3, 4, 5, 6, 7],
    [1, 3, 5, 7, 9],
    [0, 2, 4, 6, 8],
    [4, 5, 6, 7, 8],
    [5, 4, 3, 2, 1],
    [6, 5, 4, 3, 2],
    [9, 7, 5, 3, 1],
    [3, 1, 4, 1, 5],
    [2, 7, 1, 8, 2],
    [4, 4, 4, 4, 4],
], dtype=torch.float32)
# 1 if increasing, 0 if decreasing (mixed or flat also count as 0)
y = torch.tensor([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0], dtype=torch.float32).view(-1, 1)
neurons = 4

x: tensor([[1., 2., 3., 4., 5.],
        [2., 3., 4., 5., 6.],
        [3., 4., 5., 6., 7.],
        [1., 3., 5., 7., 9.],
        [0., 2., 4., 6., 8.],
        [4., 5., 6., 7., 8.],
        [5., 4., 3., 2., 1.],
        [6., 5., 4., 3., 2.],
        [9., 7., 5., 3., 1.],
        [3., 1., 4., 1., 5.],
        [2., 7., 1., 8., 2.],
        [4., 4., 4., 4., 4.]])
y: tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.]])


In [ ]:
# model declaration
class MultiLayerPerceptron(nn.Module):
    def __init__(self, input_size, output_size, hidden_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features=input_size, out_features=hidden_size),
            nn.ReLU(),
            nn.Linear(in_features=hidden_size, out_features=output_size),
        )

    def forward(self, x):
        return self.layers(x)

In [4]:
# model initialization
model = MultiLayerPerceptron(input_size=x.shape[-1], output_size=y.shape[-1], hidden_size=neurons)
for name, param in model.named_parameters():
    print(f"{name}: {param.data}")
# optimizer initialization
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)

layers.0.weight: tensor([[-0.4253, -0.1366, -0.0949, -0.3835,  0.4025],
        [ 0.2375, -0.1656,  0.3501,  0.0152,  0.1659],
        [-0.0583,  0.3093,  0.3484,  0.0294, -0.3254],
        [ 0.0160,  0.1534,  0.2432, -0.0165, -0.1307]])
layers.0.bias: tensor([ 0.0382,  0.0497,  0.4317, -0.3674])
layers.2.weight: tensor([[-0.4517, -0.3270,  0.0420,  0.4048]])
layers.2.bias: tensor([-0.0713])


In [ ]:
# training
epochs = 500
for epoch in range(epochs):
    # forward pass
    z = model(x)
    # calculate loss
    loss = nn.BCEWithLogitsLoss()(input=z, target=y)
    # backward pass
    optimizer.zero_grad()
    loss.backward()
    if epoch % 50 == 0:
        print(f"epoch {epoch} loss: {loss.item()}")
        for name, param in model.named_parameters():
            print(f"{name}.grad: {param.grad}")
    # update parameters
    optimizer.step()

epoch 0 loss: 0.8968117833137512
layers.0.weight.grad: tensor([[ 0.0000,  0.0556,  0.1112,  0.1668,  0.2224],
        [-0.1285,  0.0167,  0.2892,  0.4120,  0.6767],
        [ 0.0165, -0.0021, -0.0372, -0.0529, -0.0870],
        [ 0.1813,  0.0737, -0.1914, -0.2712, -0.5268]])
layers.0.bias.grad: tensor([ 0.0278,  0.0461, -0.0059, -0.0098])
layers.2.weight.grad: tensor([[-0.0187, -0.5773,  0.2287,  0.1103]])
layers.2.bias.grad: tensor([-0.1409])
epoch 50 loss: 0.14281199872493744
layers.0.weight.grad: tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [-0.0326,  0.0191, -0.0133,  0.0552,  0.0060],
        [-0.0412, -0.0171, -0.0056,  0.0223,  0.0169],
        [ 0.0585,  0.0151, -0.0071, -0.0570, -0.0507]])
layers.0.bias.grad: tensor([ 0.0000, -0.0081, -0.0099,  0.0125])
layers.2.weight.grad: tensor([[ 0.0000,  0.0942,  0.1142, -0.0872]])
layers.2.bias.grad: tensor([0.0190])
epoch 100 loss: 0.03652099147439003
layers.0.weight.grad: tensor([[ 0.0000,  0.0000,  0.0000,  0.0000, 

In [6]:
# prediction
with torch.inference_mode():
    z = model(x)
# activation function (value > 0 is 1, else 0)
y_pred = (z > 0).float()
print(f"y_pred: {y_pred.view(-1)}")
print(f"y:      {y.view(-1)}")

y_pred: tensor([1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.])
y:      tensor([1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.])
